In [1]:
import sys, os; sys.path.append(os.path.join(os.path.dirname("2_test_distances.ipynb"), '..'))

from sklearn.metrics import confusion_matrix, accuracy_score
from siamese.models.mobilenet_v3_large import model
from siamese.dataset.data_loader import data_loader
import numpy as np
import torch

In [2]:
PKLOT_SIAMESE_PATH = "../siamese/_models/cnrparkext_plds_to_pklot_dea6f/model_cnrparkext_plds_5c4be.pt"
PKLOT_DATASET_CONFIG = {
                        "train": [
                            "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/siamese/CNRParkEXT",
                            "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/siamese/PLds"    
                        ],
                        "test": "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/siamese/PKLot",
                        "split": 0.7,
                        "full_difficult": True,
                        "img_size": [128, 128]
                    }

CNRPARK_SIAMESE_PATH = "../siamese/_models/pklot_plds_to_cnrparkext_b5d8f/model_pklot_plds_01e83.pt"
CNRPARK_DATASET_CONFIG = {
                        "train": [
                            "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/siamese/PKLot",
                            "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/siamese/PLds"
                        ],
                        "test": "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/siamese/CNRParkEXT",
                        "split": 0.7,
                        "full_difficult": True,
                        "img_size": [128, 128]
                    }

PLDS_SIAMESE_PATH = "../siamese/_models/pklot_cnrparkext_to_plds_a404f/model_pklot_cnrparkext_2da05.pt"
PLDS_DATASET_CONFIG = {
                        "train": [
                            "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/siamese/PKLot",
                            "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/siamese/CNRParkEXT"
                        ],
                        "test": "/home/tteuz/Desktop/mestrado/meta_pklot/datasets/siamese/PLds",
                        "split": 0.7,
                        "full_difficult": True,
                        "img_size": [128, 128]
                    }

MODEL_CONFIG = { "optimizer": "Adam", "device": "cuda:0" }

In [3]:
PKLOT_THRESHOLDS = {
    "PUCPR": 0.50744,
    "UFPR04": 0.5666,
    "UFPR05": 0.56586,
    "MARGIN": 0.5891263484954834
}

CNRPARK_THRESHOLDS = {
    "CNR-CAMERA-1": 0.33219,
    "CNR-CAMERA-2": 0.48436,
    "CNR-CAMERA-3": 0.49513,
    "CNR-CAMERA-4": 0.42834,
    "CNR-CAMERA-5": 0.5959,
    "CNR-CAMERA-6": 0.49102,
    "CNR-CAMERA-7": 0.4403,
    "CNR-CAMERA-8": 0.4337,
    "CNR-CAMERA-9": 0.51647,
    "MARGIN": 0.4468725323677063
}

PLDS_THRESHOLDS = {
    "QRIDR": 0.57911,
    "ISSHK": 0.45377,
    "VXUSD": 0.52592,
    "VMLIX": 0.46843,
    "MARGIN": 0.458501935005188
}

### PKLot

In [4]:
test_model = model(pre_trained=False, config=MODEL_CONFIG)
test_model.load_state_dict(torch.load(PKLOT_SIAMESE_PATH, map_location=MODEL_CONFIG["device"]))

ds_loader = data_loader(PKLOT_DATASET_CONFIG)
test_subsets = ds_loader.get_test_subsets()

final_cm, final_cm_margin = [[0, 0], [0, 0]], [[0, 0], [0, 0]]
for key, subset in test_subsets.items():
    test_loader = torch.utils.data.DataLoader(subset, batch_size=500, shuffle=False, num_workers=8, pin_memory=True)
    preds, labels = test_model.predict(test_loader, PKLOT_THRESHOLDS[key])
    preds_margin, labels_margin = test_model.predict(test_loader, PKLOT_THRESHOLDS["MARGIN"])

    accuracy = accuracy_score(labels, preds)
    accuracy_margin = accuracy_score(labels_margin, preds_margin)
    final_cm += confusion_matrix(labels, preds)
    final_cm_margin += confusion_matrix(labels_margin, preds_margin)

    print(f"Test {key} - Accuracy: {accuracy:.4f} | Margin Accuracy: {accuracy_margin:.4f}")

print("")
print("Final Accuracy: {:.4f}".format(np.trace(final_cm) / np.sum(final_cm)))
print("Final Margin Accuracy: {:.4f}".format(np.trace(final_cm_margin) / np.sum(final_cm_margin)))

Test PUCPR - Accuracy: 0.8863 | Margin Accuracy: 0.9000
Test UFPR05 - Accuracy: 0.9280 | Margin Accuracy: 0.9294
Test UFPR04 - Accuracy: 0.9184 | Margin Accuracy: 0.9203

Final Accuracy: 0.9000
Final Margin Accuracy: 0.9093


### CNRParkEXT

In [5]:
test_model = model(pre_trained=False, config=MODEL_CONFIG)
test_model.load_state_dict(torch.load(CNRPARK_SIAMESE_PATH, map_location=MODEL_CONFIG["device"]))

ds_loader = data_loader(CNRPARK_DATASET_CONFIG)
test_subsets = ds_loader.get_test_subsets()

final_cm, final_cm_margin = [[0, 0], [0, 0]], [[0, 0], [0, 0]]
for key, subset in test_subsets.items():
    test_loader = torch.utils.data.DataLoader(subset, batch_size=500, shuffle=False, num_workers=8, pin_memory=True)
    preds, labels = test_model.predict(test_loader, CNRPARK_THRESHOLDS[key])
    preds_margin, labels_margin = test_model.predict(test_loader, CNRPARK_THRESHOLDS["MARGIN"])

    accuracy = accuracy_score(labels, preds)
    accuracy_margin = accuracy_score(labels_margin, preds_margin)
    final_cm += confusion_matrix(labels, preds)
    final_cm_margin += confusion_matrix(labels_margin, preds_margin)

    print(f"Test {key} - Accuracy: {accuracy:.4f} | Margin Accuracy: {accuracy_margin:.4f}")

print("")
print("Final Accuracy: {:.4f}".format(np.trace(final_cm) / np.sum(final_cm)))
print("Final Margin Accuracy: {:.4f}".format(np.trace(final_cm_margin) / np.sum(final_cm_margin)))

Test CNR-CAMERA-1 - Accuracy: 0.8262 | Margin Accuracy: 0.8815
Test CNR-CAMERA-2 - Accuracy: 0.8741 | Margin Accuracy: 0.8813
Test CNR-CAMERA-5 - Accuracy: 0.8138 | Margin Accuracy: 0.8678
Test CNR-CAMERA-3 - Accuracy: 0.8794 | Margin Accuracy: 0.8854
Test CNR-CAMERA-7 - Accuracy: 0.8696 | Margin Accuracy: 0.8673
Test CNR-CAMERA-6 - Accuracy: 0.8698 | Margin Accuracy: 0.8761
Test CNR-CAMERA-4 - Accuracy: 0.9016 | Margin Accuracy: 0.9013
Test CNR-CAMERA-9 - Accuracy: 0.8661 | Margin Accuracy: 0.8581
Test CNR-CAMERA-8 - Accuracy: 0.8852 | Margin Accuracy: 0.8894

Final Accuracy: 0.8642
Final Margin Accuracy: 0.8788


### PLds

In [6]:
test_model = model(pre_trained=False, config=MODEL_CONFIG)
test_model.load_state_dict(torch.load(PLDS_SIAMESE_PATH, map_location=MODEL_CONFIG["device"]))

ds_loader = data_loader(PLDS_DATASET_CONFIG)
test_subsets = ds_loader.get_test_subsets()

final_cm, final_cm_margin = [[0, 0], [0, 0]], [[0, 0], [0, 0]]
for key, subset in test_subsets.items():
    test_loader = torch.utils.data.DataLoader(subset, batch_size=500, shuffle=False, num_workers=8, pin_memory=True)
    preds, labels = test_model.predict(test_loader, PLDS_THRESHOLDS[key])
    preds_margin, labels_margin = test_model.predict(test_loader, PLDS_THRESHOLDS["MARGIN"])

    accuracy = accuracy_score(labels, preds)
    accuracy_margin = accuracy_score(labels_margin, preds_margin)
    final_cm += confusion_matrix(labels, preds)
    final_cm_margin += confusion_matrix(labels_margin, preds_margin)

    print(f"Test {key} - Accuracy: {accuracy:.4f} | Margin Accuracy: {accuracy_margin:.4f}")

print("")
print("Final Accuracy: {:.4f}".format(np.trace(final_cm) / np.sum(final_cm)))
print("Final Margin Accuracy: {:.4f}".format(np.trace(final_cm_margin) / np.sum(final_cm_margin)))

Test QRIDR - Accuracy: 0.8571 | Margin Accuracy: 0.8643
Test ISSHK - Accuracy: 0.8325 | Margin Accuracy: 0.8325
Test VXUSD - Accuracy: 0.9375 | Margin Accuracy: 0.9375
Test VMLIX - Accuracy: 0.8471 | Margin Accuracy: 0.8441

Final Accuracy: 0.8470
Final Margin Accuracy: 0.8470
